# 📚 Multiple Regression & Diagnostics: More Predictors, More Care
> Section 12 of the Statistics & A/B Testing curriculum — Part 3: Regression  
> 통계 & A/B 테스트 커리큘럼 섹션 12 — Part 3: 회귀분석

---
# 🎯 Learning Objective (학습 목표)
Today I want to learn / 오늘 배우고 싶은 것:
- [x] Fit a multiple regression and interpret a coefficient as "holding other variables constant"<br>다중회귀를 맞추고 계수를 "다른 변수를 고정했을 때"로 해석하기
- [x] Detect multicollinearity with VIF and explain why it destabilizes coefficients<br>VIF로 다중공선성을 탐지하고 계수를 불안정하게 만드는 이유 설명하기
- [x] Use Adjusted R-squared instead of plain R-squared when comparing models with different predictors, and run a full residual diagnostic checklist<br>예측변수가 다른 모델을 비교할 때 일반 R-squared 대신 Adjusted R-squared 사용하고, 완전한 잔차 진단 체크리스트 실행하기

---
# 🧠 Concept (개념)

## What is it? (이것은 무엇인가?)
*(Explain it in your own words.)*

**EN:** Multiple regression extends Section 11's simple linear regression to TWO OR MORE predictors at once: Y = intercept + slope1×X1 + slope2×X2 + ... Each coefficient now has a specific meaning — "the effect of X1 on Y, HOLDING all other predictors constant" — which is both more powerful (real business outcomes rarely have just one cause) and more fragile (predictors correlated with each other can distort the coefficients).

**KR:** 다중회귀는 섹션 11의 단순선형회귀를 두 개 이상의 예측변수로 한 번에 확장합니다: Y = 절편 + 기울기1×X1 + 기울기2×X2 + ... 이제 각 계수는 특정한 의미를 갖습니다 -- "다른 모든 예측변수를 고정했을 때 X1이 Y에 미치는 효과" -- 이는 더 강력하기도 하고(실제 비즈니스 결과는 원인이 하나뿐인 경우가 드묾) 더 취약하기도 합니다(서로 상관된 예측변수는 계수를 왜곡시킬 수 있음).

### Simple vs Multiple Regression (단순회귀 vs 다중회귀)

| | Simple (Section 11)<br>단순(섹션 11) | Multiple (Section 12)<br>다중(섹션 12) |
|---|---|---|
| Formula / 공식 | Y = b0 + b1·X1 | Y = b0 + b1·X1 + b2·X2 + ... |
| Coefficient meaning<br>계수의 의미 | "Y changes by b1 per unit of X1"<br>"X1 한 단위당 Y가 b1만큼 변함" | "...HOLDING other Xs constant"<br>"...다른 X들을 고정했을 때" |
| New risk / 새로운 위험 | — | Multicollinearity<br>다중공선성 |
| R-squared caveat<br>R-squared 주의사항 | — | Always increases with more predictors -- use Adjusted R-squared<br>예측변수가 늘면 항상 증가 -- Adjusted R-squared 사용 |

## Why do we use it? (왜 사용하는가?)
*(When is it useful?)*

**EN:** A simple regression of revenue on ad spend alone might overstate ad spend's real impact if it's secretly also capturing the effect of seasonality or headcount growth. Multiple regression lets you include those other factors explicitly and isolate what ad spend contributes ON ITS OWN — the direct fix to Section 11's causation warning and Section 5's confounder problem.

**KR:** 광고비만으로 매출을 회귀하면, 그것이 은밀히 시즌성이나 인력 증가의 효과도 함께 포착하고 있다면 광고비의 실제 영향을 과장할 수 있습니다. 다중회귀는 그런 다른 요인들을 명시적으로 포함시켜 광고비가 그 자체로 기여하는 부분을 분리할 수 있게 해줍니다 -- 섹션 11의 인과관계 경고와 섹션 5의 교란변수 문제에 대한 직접적인 해결책입니다.

## When is it used in Business Analytics? (비즈니스 분석에서 언제 사용되는가?)
*(Real-world use case)*

**EN:** Predicting revenue from ad spend WHILE controlling for seasonality, estimating a price change's effect on demand while accounting for competitor pricing, and virtually every "driver analysis" that asks which of several factors actually matters once the others are accounted for.

**KR:** 시즌성을 통제하면서 광고비로부터 매출 예측, 경쟁사 가격을 고려하면서 가격 변경이 수요에 미치는 효과 추정, 그리고 여러 요인 중 다른 것들을 통제한 후에도 실제로 중요한 요인이 무엇인지 묻는 거의 모든 "동인 분석(driver analysis)".

---
# 📝 Syntax (문법)

## Basic Syntax (기본 문법)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from patsy import dmatrix


np.random.seed(42)
ad_spend = np.random.uniform(1000, 5000, 150)
season_index = np.random.uniform(0, 10, 150)  # 0=low season, 10=peak season
revenue = ad_spend * 2.2 + season_index * 300 + np.random.normal(0, 900, 150)

df = pd.DataFrame({'ad_spend': ad_spend, 'season_index': season_index, 'revenue': revenue})
model = smf.ols('revenue ~ ad_spend + season_index', data=df).fit()

print(model.params)
print(f"R-squared: {model.rsquared:.3f}")
print(model.summary())

Intercept        -4.858021
ad_spend          2.258621
season_index    278.860674
dtype: float64
R-squared: 0.920
                            OLS Regression Results                            
Dep. Variable:                revenue   R-squared:                       0.920
Model:                            OLS   Adj. R-squared:                  0.919
Method:                 Least Squares   F-statistic:                     849.6
Date:                Fri, 06 Nov 2026   Prob (F-statistic):           1.69e-81
Time:                        18:07:38   Log-Likelihood:                -1220.8
No. Observations:                 150   AIC:                             2448.
Df Residuals:                     147   BIC:                             2457.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
----------------

## Common Variations (자주 쓰는 변형)

In [10]:
# Adjusted R-squared and coefficient p-values together
# Adjusted R-squared와 계수 p-value 함께 확인
print(f"R-squared: {model.rsquared:.3f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.3f}")
print(model.pvalues)

R-squared: 0.920
Adjusted R-squared: 0.919
Intercept       9.819722e-01
ad_spend        1.782681e-79
season_index    3.901347e-23
dtype: float64


---
# 🧪 Small Examples (예제)

## Example 1 — Multiple Regression: Isolating Ad Spend's Effect from Seasonality (광고비의 순수 효과를 시즌성에서 분리)

In [ ]:
# Business question: a simple regression showed ad spend strongly predicts
# revenue -- but is that partly just seasonality? Add season_index as a
# second predictor to isolate ad spend's own effect.
# 비즈니스 질문: 단순회귀에서 광고비가 매출을 강하게 예측했지만 -- 그중
# 일부는 그냥 시즌성 아닐까? season_index를 두 번째 예측변수로 추가해
# 광고비만의 순수한 효과를 분리.

np.random.seed(1)
n = 150
season_index = np.random.uniform(0, 10, n)
ad_spend = 1000 + season_index * 200 + np.random.normal(0, 300, n)  # ad spend ALSO rises in peak season
revenue = ad_spend * 1.8 + season_index * 250 + np.random.normal(0, 500, n)

# Simple regression: ad spend alone
df = pd.DataFrame({'ad_spend': ad_spend, 'season_index': season_index, 'revenue': revenue})
model_simple = smf.ols('revenue ~ ad_spend', data=df).fit()

# Multiple regression: ad spend + season_index
model_multi = smf.ols('revenue ~ ad_spend + season_index', data=df).fit()

print(f"Simple regression -- ad_spend coefficient: {model_simple.params['ad_spend']:.3f}")
print(f"Multiple regression -- ad_spend coefficient: {model_multi.params['ad_spend']:.3f}")
print(f"Multiple regression -- season_index coefficient: {model_multi.params['season_index']:.3f}")
print()
print(f"The simple regression overstates ad spend's effect ({model_simple.params['ad_spend']:.2f}) because")
print("it's also absorbing seasonality's effect. Once season_index is added and held constant,")
print(f"ad spend's own contribution drops to {model_multi.params['ad_spend']:.2f} per dollar --")
print("a more honest, isolated estimate.")
print(f"단순회귀는 광고비의 효과를 과장합니다({model_simple.params['ad_spend']:.2f}) -- 시즌성의")
print("효과도 함께 흡수하고 있기 때문입니다. season_index를 추가해 고정하면, 광고비")
print(f"자체의 기여는 달러당 {model_multi.params['ad_spend']:.2f}로 줄어듭니다 -- 더")
print("정직하고 분리된 추정치입니다.")

Simple regression -- ad_spend coefficient: 2.694
Multiple regression -- ad_spend coefficient: 1.739
Multiple regression -- season_index coefficient: 249.786

The simple regression overstates ad spend's effect (2.69) because
it's also absorbing seasonality's effect. Once season_index is added and held constant,
ad spend's own contribution drops to 1.74 per dollar --
a more honest, isolated estimate.
단순회귀는 광고비의 효과를 과장합니다(2.69) -- 시즌성의
효과도 함께 흡수하고 있기 때문입니다. season_index를 추가해 고정하면, 광고비
자체의 기여는 달러당 1.74로 줄어듭니다 -- 더
정직하고 분리된 추정치입니다.


## Example 2 — Multicollinearity: When Predictors Correlate With Each Other (예측변수끼리 상관될 때: 다중공선성)

In [12]:
# Business question: a model includes both "unique visitors" and "total
# visits" (nearly the same thing) as predictors of sales. Does this cause a problem?
# 비즈니스 질문: 모델에 "순 방문자 수"와 "총 방문 수"(거의 같은 것)를
# 둘 다 예측변수로 넣었습니다. 문제가 되는가?

from statsmodels.stats.outliers_influence import variance_inflation_factor

np.random.seed(2)
n = 100
unique_visitors = np.random.uniform(1000, 8000, n)
total_visits = unique_visitors * 1.4 + np.random.normal(0, 200, n)  # nearly redundant
sales = unique_visitors * 0.08 + np.random.normal(0, 40, n)

df = pd.DataFrame({'unique_visitors': unique_visitors, 'total_visits': total_visits, 'sales': sales})
model = smf.ols('sales ~ unique_visitors + total_visits', data=df).fit()

print("Coefficients:")
print(model.params)
print()
print("VIF (Variance Inflation Factor) per predictor:")
for i, col in enumerate(model.model.exog_names):
    vif = variance_inflation_factor(model.model.exog, i)
    print(f"  {col}: {vif:.1f}")

print()
print("VIF above ~5-10 signals problematic multicollinearity -- unique_visitors and")
print("total_visits are so highly correlated that the model can't reliably tell which")
print("one 'deserves credit' for predicting sales, making both coefficients unstable")
print("(small data changes could swing them wildly, even flip their sign).")
print("VIF가 약 5~10을 넘으면 문제가 되는 다중공선성을 나타냅니다 -- unique_visitors와")
print("total_visits가 너무 높게 상관되어 있어 모델이 어느 쪽이 매출 예측의 '공로를")
print("인정받아야' 하는지 신뢰성 있게 구분할 수 없고, 두 계수 모두 불안정해집니다")
print("(데이터가 조금만 바뀌어도 크게 흔들리거나 부호가 뒤집힐 수 있음).")

Coefficients:
Intercept         -12.737904
unique_visitors     0.058450
total_visits        0.016488
dtype: float64

VIF (Variance Inflation Factor) per predictor:
  Intercept: 1.0
  unique_visitors: 143.3
  total_visits: 143.3

VIF above ~5-10 signals problematic multicollinearity -- unique_visitors and
total_visits are so highly correlated that the model can't reliably tell which
one 'deserves credit' for predicting sales, making both coefficients unstable
(small data changes could swing them wildly, even flip their sign).
VIF가 약 5~10을 넘으면 문제가 되는 다중공선성을 나타냅니다 -- unique_visitors와
total_visits가 너무 높게 상관되어 있어 모델이 어느 쪽이 매출 예측의 '공로를
인정받아야' 하는지 신뢰성 있게 구분할 수 없고, 두 계수 모두 불안정해집니다
(데이터가 조금만 바뀌어도 크게 흔들리거나 부호가 뒤집힐 수 있음).


## Example 3 — Adjusted R-squared: Correcting for the Number of Predictors (예측변수 개수 보정)

In [13]:
# Business question: does adding a completely RANDOM, useless predictor
# still improve a model's R-squared? What does adjusted R-squared say?
# 비즈니스 질문: 완전히 무작위이고 쓸모없는 예측변수를 추가해도 모델의
# R-squared가 개선되는가? 수정된 R-squared는 뭐라고 하는가?

np.random.seed(13)
n = 80
useful_x = np.random.uniform(0, 50, n)
y = useful_x * 3 + np.random.normal(0, 20, n)

# Model 1: just the useful predictor
df1 = pd.DataFrame({'useful_x': useful_x, 'y': y})
model1 = smf.ols('y ~ useful_x', data=df1).fit()

# Model 2: useful predictor + THREE completely random, unrelated columns
noise1 = np.random.normal(0, 1, n)
noise2 = np.random.normal(0, 1, n)
noise3 = np.random.normal(0, 1, n)
df2 = pd.DataFrame({'useful_x': useful_x, 'noise1': noise1, 'noise2': noise2, 'noise3': noise3, 'y': y})
model2 = smf.ols('y ~ useful_x + noise1 + noise2 + noise3', data=df2).fit()

print(f"Model 1 (1 predictor):        R-squared={model1.rsquared:.4f}, Adjusted R-squared={model1.rsquared_adj:.4f}")
print(f"Model 2 (+ 3 random columns): R-squared={model2.rsquared:.4f}, Adjusted R-squared={model2.rsquared_adj:.4f}")
print()
print("Plain R-squared went UP even though the three noise columns have no real relationship")
print("with y -- R-squared mechanically can't decrease when you add ANY predictor. Adjusted")
print("R-squared applies a penalty for each added predictor, and here it correctly DROPS,")
print("revealing that the extra columns are actually hurting the model's honest quality.")
print("세 노이즈 컬럼이 y와 아무 실제 관계가 없어도 일반 R-squared는 상승했습니다 --")
print("R-squared는 어떤 예측변수를 추가하든 기계적으로 감소할 수 없습니다. Adjusted")
print("R-squared는 추가된 예측변수마다 페널티를 적용하며, 여기서는 실제로 하락해")
print("추가된 컬럼들이 모델의 정직한 품질을 오히려 해치고 있음을 드러냅니다.")

Model 1 (1 predictor):        R-squared=0.8663, Adjusted R-squared=0.8646
Model 2 (+ 3 random columns): R-squared=0.8696, Adjusted R-squared=0.8626

Plain R-squared went UP even though the three noise columns have no real relationship
with y -- R-squared mechanically can't decrease when you add ANY predictor. Adjusted
R-squared applies a penalty for each added predictor, and here it correctly DROPS,
revealing that the extra columns are actually hurting the model's honest quality.
세 노이즈 컬럼이 y와 아무 실제 관계가 없어도 일반 R-squared는 상승했습니다 --
R-squared는 어떤 예측변수를 추가하든 기계적으로 감소할 수 없습니다. Adjusted
R-squared는 추가된 예측변수마다 페널티를 적용하며, 여기서는 실제로 하락해
추가된 컬럼들이 모델의 정직한 품질을 오히려 해치고 있음을 드러냅니다.


## Example 4 — Full Residual Diagnostics: Linearity, Homoscedasticity, Normality (완전한 잔차 진단: 선형성·등분산성·정규성)

In [14]:
# Business question: run a complete diagnostic checklist on a model predicting
# delivery time from distance and package weight, before trusting it.
# 비즈니스 질문: 거리와 패키지 무게로 배송 시간을 예측하는 모델을
# 신뢰하기 전에 완전한 진단 체크리스트를 실행.

from statsmodels.stats.diagnostic import het_breuschpagan

np.random.seed(4)
n = 120
distance_km = np.random.uniform(1, 40, n)
weight_kg = np.random.uniform(0.5, 20, n)
delivery_min = 10 + distance_km * 2.1 + weight_kg * 1.5 + np.random.normal(0, 8, n)

df = pd.DataFrame({'distance_km': distance_km, 'weight_kg': weight_kg, 'delivery_min': delivery_min})
model = smf.ols('delivery_min ~ distance_km + weight_kg', data=df).fit()
residuals = model.resid

print(f"R-squared: {model.rsquared:.3f}, Adjusted R-squared: {model.rsquared_adj:.3f}")
print()

# 1) Linearity: residuals vs each predictor (Section 11's approach, extended to 2 predictors)
for col in ['distance_km', 'weight_kg']:
    corr, _ = stats.pearsonr(df[col], residuals)
    print(f"1) Linearity -- correlation between {col} and residuals: {corr:.3f} (near 0 is good)")

# 2) Homoscedasticity: Breusch-Pagan test (is residual variance constant across predictions?)
bp_stat, bp_p, _, _ = het_breuschpagan(residuals, model.model.exog)
tag_bp = "(constant variance)" if bp_p >= 0.05 else "(variance is NOT constant -- caution)"
print(f"2) Homoscedasticity -- Breusch-Pagan p-value: {bp_p:.4f} {tag_bp}")

# 3) Normality of residuals: Shapiro-Wilk test
shapiro_stat, shapiro_p = stats.shapiro(residuals)
tag_sw = "(residuals look normal)" if shapiro_p >= 0.05 else "(residuals may not be normal)"
print(f"3) Normality -- Shapiro-Wilk p-value: {shapiro_p:.4f} {tag_sw}")

print()
print("All three checks look healthy -- this model's assumptions are reasonably well met.")
print("세 가지 확인 모두 양호합니다 -- 이 모델의 가정은 합리적으로 잘 충족됩니다.")

R-squared: 0.904, Adjusted R-squared: 0.903

1) Linearity -- correlation between distance_km and residuals: 0.000 (near 0 is good)
1) Linearity -- correlation between weight_kg and residuals: 0.000 (near 0 is good)
2) Homoscedasticity -- Breusch-Pagan p-value: 0.4561 (constant variance)
3) Normality -- Shapiro-Wilk p-value: 0.2907 (residuals look normal)

All three checks look healthy -- this model's assumptions are reasonably well met.
세 가지 확인 모두 양호합니다 -- 이 모델의 가정은 합리적으로 잘 충족됩니다.


## Example 5 — Practice: Apply All Four Concepts (연습: 네 가지 개념 모두 적용하기)

Fill in every `________` blank below. The cell runs with **no errors** even if you leave blanks — it just prints a reminder instead of that part's result.

아래 `________` 빈칸을 모두 채워보세요. 빈칸을 두어도 셀은 **에러 없이** 실행되며, 그 부분의 결과 대신 안내 메시지만 출력됩니다.

In [19]:
# ============================================================
# Practice -- fill in every ________ blank, then re-run the cell
# 연습 -- 아래 ________ 빈칸을 모두 채운 뒤 셀을 다시 실행하세요
# ============================================================
BLANK = "________"

def ready(*vals):
    return all(v != BLANK for v in vals)

# ---------- Part A: multiple regression -- house price from size and age ----------
np.random.seed(5)
size_sqm = np.random.uniform(40, 200, 100)
age_years = np.random.uniform(0, 40, 100)
price = 50000 + size_sqm * 1200 - age_years * 800 + np.random.normal(0, 15000, 100)

df_a = pd.DataFrame({'size_sqm': size_sqm, 'age_years': age_years, 'price': price})
model_a = smf.ols('price ~ size_sqm + age_years', data=df_a).fit()

predict_size_a = 100   # TODO: pick a house size in sqm to predict a price for, e.g. 100
predict_age_a = 10    # TODO: pick a house age in years to predict a price for, e.g. 10

if ready(predict_size_a, predict_age_a):
    predicted_price = (model_a.params['Intercept'] + model_a.params['size_sqm'] * predict_size_a
                        + model_a.params['age_years'] * predict_age_a)
    print(f"Predicted price for a {predict_size_a}sqm, {predict_age_a}-year-old house: {predicted_price:,.0f}")
else:
    print("Part A \u270f\ufe0f  Fill in predict_size_a and predict_age_a above, then re-run.")

# ---------- Part B: VIF -- check two correlated marketing metrics ----------
np.random.seed(6)
n_b = 90
impressions = np.random.uniform(5000, 50000, n_b)
clicks = impressions * 0.03 + np.random.normal(0, 100, n_b)  # highly correlated with impressions
df_b = pd.DataFrame({'impressions': impressions, 'clicks': clicks})
X_b = dmatrix('impressions + clicks', data=df_b, return_type='dataframe')  # design matrix, incl. the Intercept column

vif_threshold_b = 5   # TODO: what VIF value is commonly used as a "concerning" threshold? (e.g. 5 or 10)

if ready(vif_threshold_b):
    for i, col in enumerate(X_b.columns):
        vif = variance_inflation_factor(X_b.values, i)
        flag = " <- HIGH" if vif > vif_threshold_b else ""
        print(f"{col}: VIF={vif:.1f}{flag}")
else:
    print("Part B \u270f\ufe0f  Fill in vif_threshold_b above, then re-run.")

# ---------- Part C: adjusted R-squared -- does a new predictor actually help? ----------
np.random.seed(7)
n_c = 70
useful_c = np.random.uniform(0, 30, n_c)
y_c = useful_c * 2 + np.random.normal(0, 15, n_c)
maybe_useful_c = np.random.normal(0, 1, n_c)  # pure noise

df_c = pd.DataFrame({'useful_c': useful_c, 'maybe_useful_c': maybe_useful_c, 'y_c': y_c})
model1_c = smf.ols('y_c ~ useful_c', data=df_c).fit()
model2_c = smf.ols('y_c ~ useful_c + maybe_useful_c', data=df_c).fit()

worth_adding_c = False   # TODO: True or False -- based on ADJUSTED R-squared (not plain
                         # R-squared), does adding maybe_useful_c actually help?

if ready(worth_adding_c):
    print(f"Model 1 adjusted R-squared: {model1_c.rsquared_adj:.4f}")
    print(f"Model 2 adjusted R-squared: {model2_c.rsquared_adj:.4f}")
    improved = model2_c.rsquared_adj > model1_c.rsquared_adj
    correct = improved == worth_adding_c
    if correct:
        print("Correct!")
    else:
        print("Not quite -- compare the two adjusted R-squared values directly.")
else:
    print("Part C \u270f\ufe0f  Fill in worth_adding_c above (True or False), then re-run.")

# ---------- Part D: diagnostics -- interpret a Breusch-Pagan result ----------
np.random.seed(8)
n_d = 100
x_d = np.random.uniform(1, 100, n_d)
y_d = x_d * 3 + np.random.normal(0, x_d * 0.5, n_d)  # heteroscedastic: variance grows with x

df_d = pd.DataFrame({'x_d': x_d, 'y_d': y_d})
model_d = smf.ols('y_d ~ x_d', data=df_d).fit()
bp_stat_d, bp_p_d, _, _ = het_breuschpagan(model_d.resid, model_d.model.exog)

assumption_met_d = False   # TODO: True or False -- given the Breusch-Pagan p-value you're
                           # about to see, is the constant-variance assumption met?

if ready(assumption_met_d):
    print(f"Breusch-Pagan p-value: {bp_p_d:.4f}")
    correct = (bp_p_d >= 0.05) == assumption_met_d
    if correct:
        print("Correct!")
    else:
        print("Not quite -- p < 0.05 means variance is NOT constant.")
else:
    print("Part D \u270f\ufe0f  Fill in assumption_met_d above (True or False), then re-run.")

Predicted price for a 100sqm, 10-year-old house: 161,366
Intercept: VIF=1.0
impressions: VIF=17.9 <- HIGH
clicks: VIF=17.9 <- HIGH
Model 1 adjusted R-squared: 0.6114
Model 2 adjusted R-squared: 0.6095
Correct!
Breusch-Pagan p-value: 0.0015
Correct!


---
# ⚠️ Common Mistakes (자주 하는 실수)

### Mistake 1 — Interpreting a Coefficient Without "Holding Other Variables Constant" (다른 변수 고정 조건 없이 계수 해석)
"Ad spend's coefficient is 1.8" is incomplete for a multiple regression — the correct statement is "ad spend's coefficient is 1.8, HOLDING season_index constant," since the coefficient's value depends on which other predictors are in the model.  
"광고비의 계수는 1.8이다"는 다중회귀에서는 불완전합니다 -- 올바른 진술은 "season_index를 고정했을 때 광고비의 계수는 1.8이다"인데, 계수의 값은 모델에 어떤 다른 예측변수가 있는지에 따라 달라지기 때문입니다.

✅ **Fix / 해결:** Always state which other predictors are being "held constant" when interpreting any single coefficient.

### Mistake 2 — Ignoring Multicollinearity and Trusting Unstable Coefficients (다중공선성을 무시하고 불안정한 계수를 신뢰)
When two predictors are highly correlated (Example 2's high VIF), the model can't cleanly separate their individual effects — the coefficients themselves become unreliable, even if the model's overall predictions are still fine.  
두 예측변수가 높게 상관되어 있으면(예제 2의 높은 VIF), 모델은 각각의 개별 효과를 깔끔하게 분리할 수 없습니다 -- 모델의 전체 예측은 여전히 괜찮을 수 있어도 계수 자체는 신뢰할 수 없게 됩니다.

✅ **Fix / 해결:** Check VIF for every predictor; if it's high (commonly >5-10), consider removing one of the correlated pair or combining them.

### Mistake 3 — Comparing Plain R-squared Across Models With Different Numbers of Predictors (예측변수 개수가 다른 모델 간 일반 R-squared 비교)
Plain R-squared mechanically increases (or at worst stays flat) every time you add ANY predictor, useful or not — comparing two models' plain R-squared to decide if a new variable helped is misleading by design.  
일반 R-squared는 어떤 예측변수든 추가할 때마다(유용하든 아니든) 기계적으로 증가하거나 최악의 경우 그대로 유지됩니다 -- 새 변수가 도움이 됐는지 판단하려고 두 모델의 일반 R-squared를 비교하는 것은 설계상 오해를 부릅니다.

✅ **Fix / 해결:** Use Adjusted R-squared (which penalizes added predictors) whenever comparing models with different numbers of variables.

### Mistake 4 — Adding Predictors Just Because More Feels Better ("더 많으면 좋다"는 이유로 예측변수 추가)
Throwing every available column into a model increases complexity, multicollinearity risk, and the temptation to over-interpret noise as signal — Example 3 showed even a genuinely random column can nudge plain R-squared up.  
사용 가능한 모든 컬럼을 모델에 던져 넣으면 복잡성, 다중공선성 위험, 그리고 노이즈를 신호로 과잉 해석하고 싶은 유혹이 늘어납니다 -- 예제 3은 완전히 무작위인 컬럼조차 일반 R-squared를 밀어올릴 수 있음을 보여줬습니다.

✅ **Fix / 해결:** Add a predictor only when there's a business reason to expect it matters, and confirm with Adjusted R-squared that it actually earns its place.

### Mistake 5 — Skipping Residual Diagnostics Because R-squared Looks Good (R-squared가 좋아 보인다고 잔차 진단 생략)
A respectable R-squared says the model captures a lot of variation, but says nothing on its own about whether the relationship is truly linear, whether variance is constant, or whether residuals are normally distributed — all three need their own dedicated check.  
그럴듯한 R-squared는 모델이 변동의 상당 부분을 포착한다는 것을 말해주지만, 관계가 정말로 선형인지, 분산이 일정한지, 잔차가 정규분포인지에 대해서는 그 자체로 아무것도 말해주지 않습니다 -- 세 가지 모두 각자의 전용 확인이 필요합니다.

✅ **Fix / 해결:** Run the full checklist (linearity, homoscedasticity, normality) from Example 4 on every model before presenting it as trustworthy.

### Mistake 6 — Reflexively Dropping Every Correlated Predictor Without Business Judgment (비즈니스 판단 없이 상관된 예측변수를 반사적으로 모두 제거)
High VIF flags a statistical problem, but the fix isn't always "delete a variable" — sometimes two correlated predictors both carry genuine, distinct business meaning, and combining or re-engineering them (rather than blindly dropping one) preserves more insight.  
높은 VIF는 통계적 문제를 표시하지만, 해결책이 항상 "변수를 삭제하는 것"은 아닙니다 -- 때로는 상관된 두 예측변수 모두 진짜로 서로 다른 비즈니스적 의미를 담고 있으며, (무작정 하나를 버리기보다) 결합하거나 재설계하는 것이 더 많은 통찰을 보존합니다.

✅ **Fix / 해결:** Understand WHY two predictors correlate before deciding whether to drop, combine, or keep both despite the VIF warning.

---
# 💡 Tips (팁)
Useful tips or shortcuts / 유용한 팁과 단축법

- `model.params['column_name']` works because a formula model labels every coefficient by its column name (the intercept is `Intercept`) -- cleaner than positional indexing once you have more than one or two predictors.<br>수식 모델은 모든 계수에 컬럼 이름을 붙이므로 model.params['컬럼명']이 작동합니다(절편은 Intercept) -- 예측변수가 하나둘 이상이 되면 위치 인덱싱보다 더 깔끔합니다.
- A VIF of 1 means no correlation with other predictors at all; a VIF of 5 means the variance of that coefficient is inflated 5x due to correlation with other predictors — it's a multiplier, not a percentage.<br>VIF가 1이면 다른 예측변수와 전혀 상관이 없다는 뜻이고, VIF가 5면 다른 예측변수와의 상관관계 때문에 그 계수의 분산이 5배 부풀려졌다는 뜻입니다 -- 백분율이 아니라 배율입니다.
- `model.summary()` (introduced in Section 11) becomes even more valuable here — it prints R-squared, Adjusted R-squared, and every coefficient's p-value and CI together in one readable block.<br>섹션 11에서 소개한 model.summary()는 여기서 더욱 유용해집니다 -- R-squared, Adjusted R-squared, 그리고 모든 계수의 p-value와 신뢰구간을 하나의 읽기 좋은 블록으로 함께 출력합니다.
- Standardizing predictors (subtracting the mean, dividing by the std dev) before fitting makes coefficients directly comparable to each other in "importance," since they're no longer distorted by each variable's original units and scale.<br>예측변수를 표준화(평균을 빼고 표준편차로 나눔)한 뒤 적합하면 계수들이 원래 단위·척도에 의해 왜곡되지 않아 서로 "중요도" 측면에서 직접 비교 가능해집니다.
- This diagnostic checklist (linearity, homoscedasticity, normality, VIF) is worth turning into a reusable function you run on every regression model going forward — exactly the kind of habit that separates a careful analyst from someone who just reads off an R-squared and moves on.<br>이 진단 체크리스트(선형성, 등분산성, 정규성, VIF)는 앞으로 모든 회귀모델에 돌려볼 재사용 가능한 함수로 만들어둘 가치가 있습니다 -- 신중한 분석가와 그저 R-squared만 읽고 넘어가는 사람을 가르는 바로 그런 습관입니다.

---
# 🔗 Related Concepts (관련 개념)

```
Simple Linear Regression (Section 11)
   fitting a line, R-squared, coefficient interpretation, residual analysis
        ↓
Multiple Regression & Diagnostics (Section 12)  ← you are here / 현재 위치
   multiple predictors, multicollinearity/VIF, adjusted R-squared, full diagnostics
        ↓
Logistic Regression (Section 13)
   binary outcomes, sigmoid function, odds ratios
        ↓
[ Part 3: Regression -- COMPLETE / Part 3: 회귀분석 -- 완료 ]
        ↓
[ Part 4: A/B Testing in Practice begins / Part 4: A/B 테스트 실무, 여기서 시작 ]
A/B Test Design -> Sample Size & Power -> Analysis -> Pitfalls
```

*How is today's topic connected to other concepts?*

**EN:** This section extends the toolkit that started with Section 11 to any business question with more than one predictor, and Section 13 closes out Part 3 by handling the case where the OUTCOME itself is binary instead of continuous: multiple regression is what simple regression becomes once a business question genuinely has more than one cause, and the diagnostic habits built across Sections 7, 9, 10, and 11 (check assumptions, don't trust one number in isolation) now come together into one full checklist. Everything from here forward in Part 4 assumes this same mindset applied to a very specific type of comparison — an A/B test is, underneath the hood, often literally a regression with a single 0/1 "treatment" predictor, and the confounder-control logic from Example 1 is exactly why randomization in a real experiment is so powerful: it makes every OTHER variable roughly balanced between groups automatically, without needing to measure and include them all by hand.

**KR:** 이 섹션은 섹션 11에서 시작된 도구 세트를 예측변수가 둘 이상인 모든 비즈니스 질문으로 확장하며, 섹션 13은 결과 자체가 연속형이 아니라 이진형인 경우를 다루며 Part 3을 마무리합니다: 다중회귀는 비즈니스 질문에 정말로 원인이 하나 이상 있을 때의 단순회귀이며, 섹션 7, 9, 10, 11에 걸쳐 만든 진단 습관(가정을 확인하라, 숫자 하나만 따로 믿지 마라)이 이제 하나의 완전한 체크리스트로 합쳐집니다. Part 4부터는 이 같은 사고방식을 매우 특정한 유형의 비교에 적용합니다 -- A/B 테스트는 사실 그 내부적으로는 종종 0/1 "처치" 예측변수 하나를 가진 회귀와 문자 그대로 같으며, 예제 1의 교란변수 통제 논리는 정확히 실제 실험에서 무작위 배정이 왜 그렇게 강력한지를 설명합니다: 다른 모든 변수를 직접 측정하고 포함할 필요 없이, 그룹 간에 자동으로 대략 균형을 맞춰주기 때문입니다.

---
# 💼 Business Example (비즈니스 예시)
*How would a Business Analyst use this?*

**Scenario / 시나리오:**

**EN:** The retention team wants to understand what drives customer churn risk score, using three metrics: support ticket count, days since last login, and account age. Build the model, check it's trustworthy, and identify which factor matters most once the others are accounted for.

**KR:** 리텐션팀이 고객 이탈 위험 점수를 이끄는 요인을 지원 티켓 수, 마지막 로그인 이후 경과일, 계정 개설 기간이라는 세 지표로 이해하고 싶어합니다. 모델을 만들고, 신뢰할 만한지 확인하고, 다른 요인들을 고려했을 때 어느 요인이 가장 중요한지 파악합니다.

**To-do / 할 일:**
- [x] Fit the multiple regression with all three predictors<br>세 예측변수 모두로 다중회귀 맞추기
- [x] Run the full diagnostic checklist (linearity, VIF, homoscedasticity, normality)<br>완전한 진단 체크리스트 실행하기(선형성, VIF, 등분산성, 정규성)
- [x] Interpret each coefficient with proper "holding constant" language<br>각 계수를 올바른 "고정했을 때" 표현으로 해석하기
- [x] Recommend where the retention team should focus<br>리텐션팀이 어디에 집중해야 할지 제안하기

In [16]:
np.random.seed(70)
n = 200
tickets = np.random.poisson(3, n)
days_since_login = np.random.exponential(10, n)
account_age_months = np.random.uniform(1, 60, n)

churn_risk = (tickets * 3 + days_since_login * 0.8 - account_age_months * 0.3
              + np.random.normal(0, 8, n) + 20)

df = pd.DataFrame({
    'tickets': tickets, 'days_since_login': days_since_login, 'account_age_months': account_age_months,
    'churn_risk': churn_risk
})
model = smf.ols('churn_risk ~ tickets + days_since_login + account_age_months', data=df).fit()

print("Coefficients (holding other variables constant):")
print(model.params.round(3))
print()
print(f"R-squared: {model.rsquared:.3f}, Adjusted R-squared: {model.rsquared_adj:.3f}")
print()

print("Diagnostic checklist:")
for col in ['tickets', 'days_since_login', 'account_age_months']:
    corr, _ = stats.pearsonr(df[col], model.resid)
    print(f"  Linearity ({col}): correlation with residuals = {corr:.3f}")

for i, col in enumerate(model.model.exog_names):
    vif = variance_inflation_factor(model.model.exog, i)
    print(f"  VIF ({col}): {vif:.2f}")

bp_stat, bp_p, _, _ = het_breuschpagan(model.resid, model.model.exog)
print(f"  Homoscedasticity (Breusch-Pagan p-value): {bp_p:.4f}")

shapiro_stat, shapiro_p = stats.shapiro(model.resid)
print(f"  Normality (Shapiro-Wilk p-value): {shapiro_p:.4f}")

print()
print("Recommendation: diagnostics look reasonable. Support ticket count has the largest")
print("effect on churn risk once login recency and account age are held constant --")
print("prioritize proactive outreach for customers with rising ticket counts.")
print()
print("제안: 진단 결과 양호합니다. 로그인 최근성과 계정 기간을 고정했을 때, 지원 티켓")
print("수가 이탈 위험에 가장 큰 영향을 미칩니다 -- 티켓 수가 늘고 있는 고객에 대한")
print("선제적 접촉을 우선하세요.")

Coefficients (holding other variables constant):
Intercept             19.949
tickets                2.968
days_since_login       0.803
account_age_months    -0.309
dtype: float64

R-squared: 0.670, Adjusted R-squared: 0.665

Diagnostic checklist:
  Linearity (tickets): correlation with residuals = 0.000
  Linearity (days_since_login): correlation with residuals = -0.000
  Linearity (account_age_months): correlation with residuals = 0.000
  VIF (Intercept): 1.00
  VIF (tickets): 1.04
  VIF (days_since_login): 1.04
  VIF (account_age_months): 1.00
  Homoscedasticity (Breusch-Pagan p-value): 0.8209
  Normality (Shapiro-Wilk p-value): 0.0987

Recommendation: diagnostics look reasonable. Support ticket count has the largest
effect on churn risk once login recency and account age are held constant --
prioritize proactive outreach for customers with rising ticket counts.

제안: 진단 결과 양호합니다. 로그인 최근성과 계정 기간을 고정했을 때, 지원 티켓
수가 이탈 위험에 가장 큰 영향을 미칩니다 -- 티켓 수가 늘고 있는 고객에 대한
선제적 접촉을 우선하세요.


---
# 📝 Summary (요약)
*Write today's concept in 3~5 sentences.*

**EN:** Multiple regression extends simple regression to two or more predictors, letting each coefficient answer "how much does Y change per unit of this X, holding every other X constant" — a direct tool for isolating one variable's true effect from confounders, exactly what Section 5's correlation and Section 11's simple regression couldn't do alone. Multicollinearity (checked via VIF) is the main new risk: when predictors correlate with each other, individual coefficients become unstable even if overall predictions stay fine. Plain R-squared mechanically increases with every added predictor regardless of whether it helps, so comparing models fairly requires Adjusted R-squared, which penalizes unnecessary complexity. This section closes Part 3 by assembling a complete diagnostic checklist — linearity (residuals vs. each predictor), homoscedasticity (Breusch-Pagan test), normality of residuals (Shapiro-Wilk test), and multicollinearity (VIF) — the full set of assumption checks every model deserves before its coefficients are trusted or reported.

**KR:** 다중회귀는 단순회귀를 두 개 이상의 예측변수로 확장해, 각 계수가 "다른 모든 X를 고정했을 때 이 X 한 단위당 Y가 얼마나 변하는가"에 답하게 합니다 -- 섹션 5의 상관관계와 섹션 11의 단순회귀 혼자서는 할 수 없었던, 교란변수로부터 한 변수의 진짜 효과를 분리하는 직접적인 도구입니다. 다중공선성(VIF로 확인)이 주된 새 위험입니다: 예측변수가 서로 상관되면, 전체 예측은 여전히 괜찮아도 개별 계수는 불안정해집니다. 일반 R-squared는 도움이 되든 안 되든 예측변수를 추가할 때마다 기계적으로 증가하므로, 모델을 공정하게 비교하려면 불필요한 복잡성에 페널티를 주는 Adjusted R-squared가 필요합니다. 이 섹션은 완전한 진단 체크리스트 -- 선형성(각 예측변수 대 잔차), 등분산성(Breusch-Pagan 검정), 잔차의 정규성(Shapiro-Wilk 검정), 다중공선성(VIF) -- 를 조립하며 Part 3을 마무리합니다. 이는 모든 모델이 계수를 신뢰받거나 보고되기 전에 받아야 할 완전한 가정 확인 세트입니다.

---
# 📌 One Sentence Summary (한 문장 요약)
Today's topic in ONE sentence. / 오늘의 주제를 한 문장으로.

> **EN:** Multiple regression isolates each predictor's effect while holding the others constant, but that power comes with new risks — multicollinearity (checked via VIF) can destabilize coefficients, plain R-squared always rewards adding predictors regardless of usefulness (use Adjusted R-squared instead), and every model deserves a full diagnostic checklist (linearity, homoscedasticity, normality) before its results are trusted.

> **KR:** 다중회귀는 다른 예측변수를 고정한 채 각 예측변수의 효과를 분리하지만, 그 힘에는 새로운 위험이 따릅니다 -- 다중공선성(VIF로 확인)이 계수를 불안정하게 만들 수 있고, 일반 R-squared는 유용성과 무관하게 예측변수 추가를 항상 보상하며(대신 Adjusted R-squared 사용), 모든 모델은 결과가 신뢰받기 전에 완전한 진단 체크리스트(선형성, 등분산성, 정규성)를 거쳐야 합니다.

---
# ❓ Review Questions (복습 질문)

**Q1 (EN).** In a model predicting revenue from both ad spend and season_index, what does it mean to say "ad spend's coefficient is 1.8"? What's missing from that statement?  
**Q1 (KR).** 광고비와 season_index 둘 다로 매출을 예측하는 모델에서, "광고비의 계수는 1.8이다"라는 말은 무엇을 의미하나요? 그 진술에서 빠진 것은 무엇인가요?

Holding season_index constant, one extra $1 of ad spend is associated with 1.8 more revenue. Missing: units and a confidence interval (and it is an association, not proven causation).  
season_index를 고정했을 때 광고비 $1 증가당 매출이 1.8 늘어난다는 뜻입니다. 빠진 것은 단위와 신뢰구간입니다. 인과가 아니라 연관이라는 점도 말해야 합니다.

**Q2 (EN).** Two predictors have a VIF of 18 each. What does this number mean, and what should you consider doing about it?  
**Q2 (KR).** 두 예측변수의 VIF가 각각 18입니다. 이 숫자는 무엇을 의미하며, 이에 대해 무엇을 고려해야 하나요?

VIF 18 means severe multicollinearity. Each predictor is largely explained by the other, so coefficients and standard errors become unstable. Consider dropping one, combining them, or using regularization.    
VIF 18은 심한 다중공선성입니다. 서로가 서로를 거의 설명해서 계수와 표준오차가 불안정합니다. 하나를 제거하거나 합치거나 정규화를 고려합니다.

**Q3 (EN).** Why can plain R-squared never decrease when you add a new predictor, even a completely useless one, and what should you check instead?  
**Q3 (KR).** 완전히 쓸모없는 예측변수를 추가해도 일반 R-squared는 왜 절대 감소하지 않으며, 대신 무엇을 확인해야 하나요?

Adding a predictor can only reduce residual error or leave it unchanged, so plain R² never drops. Check adjusted R², which penalizes extra predictors.  
예측변수를 더하면 잔차 오차는 줄거나 그대로라서 R²는 줄지 않습니다. 변수 개수에 벌점을 주는 조정 R²를 확인합니다.

**Q4 (EN).** A Breusch-Pagan test on a model's residuals returns p=0.001. What assumption is being tested, and what does this result suggest?  
**Q4 (KR).** 모델 잔차에 대한 Breusch-Pagan 검정 결과 p=0.001이 나왔습니다. 어떤 가정을 검정하고 있으며, 이 결과는 무엇을 시사하나요?

It tests homoscedasticity, meaning constant residual variance. p=0.001 suggests heteroscedasticity, so standard errors and p-values may be unreliable. Consider robust standard errors or a transform.  
등분산성(잔차 분산이 일정한지)을 검정합니다. p=0.001이면 이분산성이 의심되어 표준오차와 p-value를 믿기 어렵습니다. 강건 표준오차나 변환을 고려합니다.

**Q5 (EN).** Name the four checks that make up a full regression diagnostic checklist, and what each one verifies.  
**Q5 (KR).** 완전한 회귀 진단 체크리스트를 구성하는 네 가지 확인 항목을 말하고, 각각 무엇을 검증하는지 설명하세요.

(1) Linearity: residuals vs fitted show no curve. (2) Independence: residuals are not correlated, for example Durbin-Watson. (3) Homoscedasticity: residual variance is constant. (4) Normality: residuals are roughly normal, for example a Q-Q plot.  
(1) 선형성: 잔차 대 예측값에 곡선이 없는지. (2) 독립성: 잔차끼리 상관이 없는지(Durbin-Watson 등). (3) 등분산성: 잔차 분산이 일정한지. (4) 정규성: 잔차가 대략 정규인지(Q-Q 플롯 등).

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*
*➡️ Next up: Logistic Regression (Section 13), the last stop in Part 3 before A/B Testing begins. / 다음: 로지스틱회귀(섹션 13), Part 3의 마지막 정거장이며 그 다음이 A/B 테스트입니다.*